# Lab: Deploy Strands-Orchestrated Financial Assistant (HITL) to Amazon Bedrock AgentCore

This lab deploys a Strands-orchestrated multi-agent financial assistant with **Human-in-the-Loop (HITL)** to Amazon Bedrock AgentCore.

The orchestrator agent coordinates two specialized agents:
- **Budget Agent**: Analyzes spending and creates budget recommendations
- **Financial Analysis Agent**: Provides investment portfolio recommendations

**HITL Pattern**: Unlike the basic strands-orchestrator, this version asks for user confirmation before proceeding from budget analysis to investment recommendations. The conversation loop runs on the client side, with each turn invoking the AgentCore endpoint.

**Important**: For HITL to work, subsequent requests must be handled by the same container so that conversation state persists. This is best-effort in the happy path.

## Architecture

```
Client (conversation loop)
    │
    ├─── Turn 1: "Create financial plan" ───┐
    │                                       │
    │◄── Budget analysis + "confirm?" ──────┤
    │                                       │
    ├─── Turn 2: "Invest $400" ─────────────┤
    │                                       ▼
    │◄── Portfolio recommendation ───── AgentCore
                                        (Orchestrator Agent)
                                            │
                                    ┌───────┴───────┐
                                    ▼               ▼
                               Budget Agent   Financial Agent
```

## What You Will Learn

You'll learn how to leverage AgentCore's purpose-built infrastructure for running agents at scale while maintaining security, performance, and reliability standards required for enterprise applications.

## Amazon Bedrock AgentCore Runtime

[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) provides a secure, serverless, and purpose-built hosting environment for deploying and running AI agents or tools, shortening the time to value from experiments to production-grade agents.

[Learn more about how AgentCore Runtime works](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html)

## Amazon Bedrock AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) helps you trace, debug, and monitor agent performance in production environments. It offers detailed visualizations of each step in the agent workflow, enabling you to inspect an agent's execution path, audit intermediate outputs, and debug performance bottlenecks and failures.

### Step 1: Install dependencies and aws configs

In [1]:
# Install dependencies for AgentCore deployment
!pip install --force-reinstall -U -r requirements.txt --quiet


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [2]:
!aws configure set sso_start_url https://temporal.awsapps.com/start --profile corp-sso
!aws configure set sso_region us-west-2 --profile corp-sso
!aws configure set sso_account_id 269172689222 --profile corp-sso
!aws configure set sso_role_name AWSAdministratorAccess --profile corp-sso
!aws configure set region us-west-2 --profile corp-sso
!aws configure set output json --profile corp-sso

In [3]:
!aws sso login --profile corp-sso

Attempting to automatically open the SSO authorization page in your default browser.
If the browser does not open or you wish to use a different device to authorize this request, open the following URL:

https://temporal.awsapps.com/start/#/device

Then enter the code:

KQMM-GXQM
Successfully logged into Start URL: https://temporal.awsapps.com/start


In [4]:
import os, boto3
os.environ["AWS_PROFILE"]="corp-sso"
os.environ["AWS_REGION"]="us-west-2"          # your service region
os.environ["AWS_DEFAULT_REGION"]="us-west-2"
os.environ["AWS_SDK_LOAD_CONFIG"]="1"
boto3.Session().client("sts").get_caller_identity()

{'UserId': 'AROAT5K7RJFDAJL6SBISO:cornelia.davis@temporal.io',
 'Account': '269172689222',
 'Arn': 'arn:aws:sts::269172689222:assumed-role/AWSReservedSSO_AWSAdministratorAccess_649b0cd59d029bb2/cornelia.davis@temporal.io',
 'ResponseMetadata': {'RequestId': '98d8551a-1da3-41de-bd74-bff0b63a367f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '98d8551a-1da3-41de-bd74-bff0b63a367f',
   'x-amz-sts-extended-request-id': 'MTp1cy13ZXN0LTI6UzoxNzY5OTc1Njc2NTU5OlI6V1RodnhnSTY=',
   'content-type': 'text/xml',
   'content-length': '513',
   'date': 'Sun, 01 Feb 2026 19:54:36 GMT'},
  'RetryAttempts': 0}}

In [5]:
# Import AgentCore Runtime deployment tools and utilities
import uuid
from utils import setup_cognito_user_pool, reauthenticate_user
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
from typing import Any, Optional
import urllib.parse
import requests
import json

In [6]:
# Initialize AWS session and get current region
boto_session = Session()
region = boto_session.region_name
print(region)

us-west-2


### Step 2: Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed agent server.

In [7]:
# Reload the utils.agentcore_utils submodule to pick up changes
# I've reorded the above cells so that this shouldn't be needed in the future.
import importlib
from utils import agentcore_utils
importlib.reload(agentcore_utils)

# Also reload utils to refresh the imports
import utils
importlib.reload(utils)

# Re-import the function
from utils import setup_cognito_user_pool, reauthenticate_user

print("Module reloaded successfully")

Module reloaded successfully


In [8]:
# Set up Amazon Cognito for AgentCore Runtime authentication
print("Setting up Amazon Cognito user pool...")

cognito_config = setup_cognito_user_pool()

print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

Setting up Amazon Cognito user pool...
AWS_PROFILE: corp-sso
AWS_REGION: us-west-2
Pool id: us-west-2_9FDRAUiTs
Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_9FDRAUiTs/.well-known/openid-configuration
Client ID: 35dvbvpsut487ftdqoh2r7l6dh
Bearer Token: eyJraWQiOiJIY0xFMHI1elwva0lFYVJrWW8xamJMNHZRWTFSR1pIM29RZVBSZnoyQXJNTT0iLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiJjODgxODM4MC0wMGUxLTcwNWQtNDIyNi0xODlhZjY3ODhjMmUiLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAudXMtd2VzdC0yLmFtYXpvbmF3cy5jb21cL3VzLXdlc3QtMl85RkRSQVVpVHMiLCJjbGllbnRfaWQiOiIzNWR2YnZwc3V0NDg3ZnRkcW9oMnI3bDZkaCIsIm9yaWdpbl9qdGkiOiJjNmU0MTYxMS0zMzE5LTRjNDEtOGJiYS05MjA1NjBlYjY4ZjUiLCJldmVudF9pZCI6ImRiYzA4ZDJkLTM0MjgtNDg3OC1iNTE3LWNhOGNlN2ZhN2RiMSIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoiYXdzLmNvZ25pdG8uc2lnbmluLnVzZXIuYWRtaW4iLCJhdXRoX3RpbWUiOjE3Njk5NzU3MDIsImV4cCI6MTc2OTk3OTMwMiwiaWF0IjoxNzY5OTc1NzAyLCJqdGkiOiI0ZWVmZmNjZC01YjBhLTQyNjAtODI1Ny1mNDVlNWQyNmYxMmIiLCJ1c2VybmFtZSI6InRlc3R1c2VyIn0.MKN6EoisJPihDt-Tk3Ekuz-0E53K_DW

In [9]:
# Configure JWT authorization for AgentCore Runtime
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

print(auth_config)

{'customJWTAuthorizer': {'allowedClients': ['35dvbvpsut487ftdqoh2r7l6dh'], 'discoveryUrl': 'https://cognito-idp.us-west-2.amazonaws.com/us-west-2_9FDRAUiTs/.well-known/openid-configuration'}}


## Strands Orchestrator Deployment

### Step 3: Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables, and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as a container and push to ECR using CI/CD pipelines and IaC.

In this tutorial, we will use the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore Runtime.

### Step 3.1: Configure AgentCore Runtime deployment

First, we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created, and a requirements file. We will also configure the starter kit to auto-create the Amazon ECR repository on launch.

During the configure step, your Dockerfile will be generated based on your application code.

In [10]:
# Configure AgentCore Runtime deployment settings
agentcore_runtime = Runtime()

agent_name = "strands_orchestrated_financial_assistant_hitl"

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="worker.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config,
)

print("Configuration completed ✓")

Entrypoint parsed: file=/Users/cdavisafc/Projects/Temporal/AI/AWS/amazon-bedrock-temporal-samples/finance-personal-assistant/strands-orchestrator-hitl/worker.py, bedrock_agentcore_name=worker
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: strands_orchestrated_financial_assistant_hitl


Configuring AgentCore Runtime...


Memory disabled
Network mode: PUBLIC
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/cdavisafc/Projects/Temporal/AI/AWS/amazon-bedrock-temporal-samples/finance-personal-assistant/strands-orchestrator-hitl/.dockerignore
Setting 'strands_orchestrated_financial_assistant_hitl' as default agent
Bedrock AgentCore configured: /Users/cdavisafc/Projects/Temporal/AI/AWS/amazon-bedrock-temporal-samples/finance-personal-assistant/strands-orchestrator-hitl/.bedrock_agentcore.yaml


Configuration completed ✓


### Step 3.2: Launching agent to AgentCore Runtime

Now that we have a Dockerfile, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

### Step 3.2: Launching agent to AgentCore Runtime

Now that we have a Dockerfile, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [11]:
# Deploy agent to AgentCore Runtime (creates ECR repo and runtime)
print("Launching Strands Orchestrator to AgentCore Runtime...")
print("This may take several minutes...")

launch_result = agentcore_runtime.launch(
    env_vars={"OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations"}
)

print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'strands_orchestrated_financial_assistant_hitl' to account 269172689222 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_orchestrated_financial_assistant_hitl


Launching Strands Orchestrator to AgentCore Runtime...
This may take several minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-strands_orchestrated_financial_assistant_hitl


ECR repository available: 269172689222.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-strands_orchestrated_financial_assistant_hitl
Getting or creating execution role for agent: strands_orchestrated_financial_assistant_hitl
Using AWS region: us-west-2, account ID: 269172689222
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-21eeb48849
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-21eeb48849
Starting execution role creation process for agent: strands_orchestrated_financial_assistant_hitl
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-west-2-21eeb48849
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-21eeb48849
✓ Role created: arn:aws:iam::269172689222:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-21eeb48849
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-strands_orchestrated_financial_assistant_hitl
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:

Launch completed ✓
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:269172689222:runtime/strands_orchestrated_financial_assistant_hitl-KI7w0D92IK
Agent ID: strands_orchestrated_financial_assistant_hitl-KI7w0D92IK


### Update runtime settings

The following code block will allow you to set a new container idle timeout. The default is 15 mins but for demo purposes, where we want to show the lifecycle events of the container, we want to shorten it.

In [ ]:
import boto3
import json
import os

# -------------------------
# CONFIGURE HERE
REGION = os.getenv("AWS_REGION", "us-west-2")
NEW_IDLE_TIMEOUT = 60   # seconds.
# -------------------------

control = boto3.client("bedrock-agentcore-control", region_name=REGION)

# ---- Step 1: List runtimes ----
print("📋 Available AgentCore runtimes:\n")

runtimes = []
next_token = None

while True:
    kwargs = {}
    if next_token:
        kwargs["nextToken"] = next_token

    resp = control.list_agent_runtimes(**kwargs)
    runtimes.extend(resp.get("agentRuntimes", []))
    next_token = resp.get("nextToken")
    if not next_token:
        break

if not runtimes:
    print("⚠️ No runtimes found in this region.")
    raise SystemExit

for i, rt in enumerate(runtimes, start=1):
    print(f"{i}. name={rt.get('name')}  id={rt.get('agentRuntimeId')}  arn={rt.get('agentRuntimeArn')}")

choice = input("\nEnter the NAME or ID of the runtime to update: ").strip()

# ---- Step 2: Select the runtime ----
selected = None
for rt in runtimes:
    if choice == rt.get("name") or choice == rt.get("agentRuntimeId"):
        selected = rt
        break

if not selected:
    print(f"❌ Runtime '{choice}' not found.")
    raise SystemExit

runtime_id = selected["agentRuntimeId"]
print(f"\n✔ Selected runtime:\n  name={selected.get('name')}\n  id={runtime_id}\n")

# ---- Step 3: Fetch current config ----
resp = control.get_agent_runtime(agentRuntimeId=runtime_id)

# Some SDK versions nest config under "agentRuntime"
runtime_cfg = resp.get("agentRuntime", resp)

print("Current lifecycleConfiguration:")
print(json.dumps(runtime_cfg.get("lifecycleConfiguration", {}), indent=2))

current_lifecycle = runtime_cfg.get("lifecycleConfiguration", {})
new_lifecycle = {
    "idleRuntimeSessionTimeout": NEW_IDLE_TIMEOUT,
    "maxLifetime": current_lifecycle.get("maxLifetime", 28800),
}

print("\n🔧 Updated lifecycleConfiguration will be:")
print(json.dumps(new_lifecycle, indent=2))

confirm = input("\nProceed with update? [y/N]: ").strip().lower()
if confirm == "y":

    # ---- Step 4: Update the runtime ----
    # Build update parameters, only including optional fields if they have valid values
    update_params = {
        "agentRuntimeId": runtime_id,
        "agentRuntimeArtifact": runtime_cfg["agentRuntimeArtifact"],
        "roleArn": runtime_cfg["roleArn"],
        "networkConfiguration": runtime_cfg["networkConfiguration"],
        "protocolConfiguration": runtime_cfg["protocolConfiguration"],
        "lifecycleConfiguration": new_lifecycle,
        "environmentVariables": runtime_cfg.get("environmentVariables", {}),
    }

    # Only include description if it exists and is not empty (API requires min length 1)
    description = runtime_cfg.get("description", "")
    if description:
        update_params["description"] = description

    # Only include authorizerConfiguration if it exists and is not empty
    authorizer_config = runtime_cfg.get("authorizerConfiguration", {})
    if authorizer_config:
        update_params["authorizerConfiguration"] = authorizer_config

    # Only include requestHeaderConfiguration if it exists and is not empty
    # (API requires at least one key to be set, e.g., requestHeaderAllowlist)
    request_header_config = runtime_cfg.get("requestHeaderConfiguration", {})
    if request_header_config:
        update_params["requestHeaderConfiguration"] = request_header_config

    update_resp = control.update_agent_runtime(**update_params)

    print("\n✅ Runtime updated successfully.\nResponse:")

else:
    print("🚫 Update cancelled but reporting current config")
    update_resp = control.get_agent_runtime(agentRuntimeId=runtime_id)

# Convert response to JSON-serializable format (handle datetime objects)
def json_serial(obj):
    """JSON serializer for objects not serializable by default json code"""
    from datetime import datetime, date
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

print(json.dumps(update_resp, indent=2, default=json_serial))


### Step 4: Invoking AgentCore Runtime (Multi-turn Conversation)

The HITL pattern requires multiple invocations to complete a financial plan:
1. First request: Ask for financial plan → Agent returns budget analysis and asks for confirmation
2. Second request: Confirm investment amount → Agent returns portfolio recommendation

**Important**: Use the same session_id for all turns in a conversation to help route to the same container.

In [ ]:
# Authenticate user and get bearer token for API access
bearer_token = reauthenticate_user(client_id=cognito_config["client_id"])

In [ ]:
def invoke_endpoint_worker(
    agent_arn: str,
    payload,
    session_id: str,
    bearer_token: Optional[str],
    region: str = "us-west-2",
    endpoint_name: str = "DEFAULT",
) -> Any:
    """Invoke agent endpoint using HTTP request with bearer token."""
    escaped_arn = urllib.parse.quote(agent_arn, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    try:
        body = json.loads(payload) if isinstance(payload, str) else payload
    except json.JSONDecodeError:
        body = {"payload": payload}

    try:
        response = requests.post(
            url,
            params={"qualifier": endpoint_name},
            headers=headers,
            json=body,
            timeout=100,
        )
        return response.text

    except requests.exceptions.RequestException as e:
        print("Failed to invoke agent endpoint: %s", str(e))
        raise

In [ ]:
# Generate a session ID for this conversation (use same ID for all turns)
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

# Turn 1: Request financial plan
print("=" * 60)
print("TURN 1: Requesting financial plan")
print("=" * 60)

response1 = invoke_endpoint_worker(
    agent_arn=launch_result.agent_arn,
    payload={
        "query": "Create a financial plan for someone earning $6000/month with $800 in dining expenses"
    },
    session_id=session_id,
    bearer_token=bearer_token,
)
print(response1)
print("\n")

# Turn 2: Confirm investment amount (the agent should ask for this)
print("=" * 60)
print("TURN 2: Confirming investment amount")
print("=" * 60)

response2 = invoke_endpoint_worker(
    agent_arn=launch_result.agent_arn,
    payload={
        "query": "Yes, please invest $400 with a moderate risk portfolio"
    },
    session_id=session_id,
    bearer_token=bearer_token,
)
print(response2)

# Print values needed for the CLI client
print("\n" + "=" * 60)
print("To use the CLI client, run:")
print("=" * 60)
print(f"export AGENT_ARN='{launch_result.agent_arn}'")
print(f"export COGNITO_CLIENT_ID='{cognito_config['client_id']}'")
print("python client.py")